# Long-Term Memory with Mem0

**Mem0** is a dedicated long-term memory layer for AI agents — available as a **managed platform** (`MemoryClient`) or **open-source** (`Memory`). It handles extraction, deduplication, vector search, and entity scoping so you do not build memory infrastructure from scratch.

```
  Conversation turn                    Mem0
  ─────────────────                    ────
  User: "I'm vegetarian, no nuts"  →  add(messages, user_id="alice")
                                       └─ infer=True → extracts facts
  New session next week            →  search("dietary?", filters={"user_id":"alice"})
                                       └─ hybrid retrieval (semantic + keyword)
```

| Concern | Mem0 approach |
|---------|---------------|
| Scope memories | `user_id`, `agent_id`, `run_id`, `metadata` |
| Store from chat | `add(messages, user_id=...)` |
| Retrieve | `search(query, filters={...})` |
| Raw vs inferred | `infer=False` skips LLM extraction |

This notebook builds **ShopCo Support** with Mem0 patterns:

1. Offline `ShopCoMem0` client mirroring the v3 API
2. `add` / `search` / `get_all` with entity filters
3. LangGraph agent that loads and persists memories each turn
4. Cross-thread recall and `agent_id` isolation
5. Mem0 vs LangMem comparison

Offline by default; optional live Mem0 OSS at the end.

In [ ]:
"""
ShopCo Support — Mem0 long-term memory lab.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Annotated, Any, Literal

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False
SHOPCO_AGENT_ID = "shopco-support-v1"


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


POLICY_SNIPPETS = {
    "returns": "Returns within 30 days. Refunds in 5-7 business days. [pol-returns-01]",
    "shipping": "Free shipping over $50. [pol-shipping-02]",
    "billing": "Charges appear as SHOPCO* on statements. [pol-billing-03]",
}

ORDERS_DB = {
    "ORD-1001": {"order_id": "ORD-1001", "status": "shipped", "email": "alice@shopco.com"},
    "ORD-1002": {"order_id": "ORD-1002", "status": "processing", "email": "bob@shopco.com"},
}

print("ShopCo Mem0 environment ready.")

---

## 1. Mem0 vs LangMem — When to Use Which?

| Dimension | **LangMem** | **Mem0** |
|-----------|-------------|----------|
| Integration | Native LangGraph `BaseStore` | Standalone SDK; LangGraph via custom nodes |
| Hosting | Self-hosted store + your embeddings | OSS self-host **or** managed platform |
| Extraction | `create_memory_manager` (you configure) | Built-in inference pipeline on `add()` |
| Retrieval | Store search + LangMem tools | Hybrid semantic + BM25 + entity linking (v3) |
| Best fit | Already on LangGraph, want tight coupling | Want managed memory or framework-agnostic LTM |

Many production systems use **checkpointer (thread) + Mem0 or LangMem (user)** together.

---

## 2. Entity Scoping — `user_id`, `agent_id`, `run_id`

Mem0 scopes every memory to entities:

- **`user_id`** — the customer (Alice, Bob)
- **`agent_id`** — which agent wrote the memory (support bot vs sales bot)
- **`run_id`** — optional session identifier

**v3 API rule:** pass entity IDs as **top-level kwargs on `add()`**, but inside **`filters`** for `search()` and `get_all()`:

```python
mem0.add(messages, user_id="alice", agent_id="shopco-support")
mem0.search("preferences", filters={"user_id": "alice", "agent_id": "shopco-support"})
```

---

## 3. Offline ShopCoMem0 — Mirrors v3 API

Runs without API keys. Swaps for `from mem0 import Memory` when `USE_LIVE_LLM=True`.

In [ ]:
@dataclass
class Mem0Record:
    id: str
    memory: str
    user_id: str
    agent_id: str | None = None
    run_id: str | None = None
    metadata: dict[str, Any] = field(default_factory=dict)
    created_at: str = ""


def _tokenize(text: str) -> set[str]:
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 2}


def _infer_facts(messages: list[dict[str, str]]) -> list[str]:
    """Rule-based stand-in for Mem0 infer=True extraction."""
    facts: list[str] = []
    for msg in messages:
        if msg.get("role") != "user":
            continue
        text = msg["content"]
        if "prefer" in text.lower():
            facts.append(f"User preference: {text}")
        elif "remember" in text.lower():
            facts.append(f"User asked to remember: {text}")
        elif re.search(r"ORD-[0-9]{4}", text.upper()):
            facts.append(f"User mentioned order {re.search(r'ORD-[0-9]{4}', text.upper()).group(0)}")
        elif "vegetarian" in text.lower() or "allerg" in text.lower():
            facts.append(f"Dietary restriction: {text}")
        elif "concise" in text.lower():
            facts.append("User prefers concise answers")
    return facts


class ShopCoMem0:
    """Offline Mem0-compatible client for ShopCo demos."""

    def __init__(self) -> None:
        self._records: dict[str, Mem0Record] = {}

    def add(
        self,
        messages: str | dict[str, str] | list[dict[str, str]],
        *,
        user_id: str,
        agent_id: str | None = None,
        run_id: str | None = None,
        metadata: dict[str, Any] | None = None,
        infer: bool = True,
    ) -> dict[str, Any]:
        if isinstance(messages, str):
            messages = [{"role": "user", "content": messages}]
        elif isinstance(messages, dict):
            messages = [messages]

        events: list[dict[str, str]] = []
        if infer:
            texts = _infer_facts(messages)
        else:
            texts = [m["content"] for m in messages if m.get("content")]

        for text in texts:
            mid = str(uuid.uuid4())
            self._records[mid] = Mem0Record(
                id=mid, memory=text, user_id=user_id, agent_id=agent_id,
                run_id=run_id, metadata=metadata or {}, created_at=utc_now(),
            )
            events.append({"id": mid, "event": "ADD", "memory": text})
        return {"results": events}

    def search(
        self,
        query: str,
        *,
        filters: dict[str, str] | None = None,
        top_k: int = 10,
        threshold: float = 0.0,
    ) -> list[dict[str, Any]]:
        filters = filters or {}
        q_tokens = _tokenize(query)
        scored: list[tuple[float, Mem0Record]] = []
        for rec in self._records.values():
            if filters.get("user_id") and rec.user_id != filters["user_id"]:
                continue
            if filters.get("agent_id") and rec.agent_id != filters["agent_id"]:
                continue
            overlap = len(q_tokens & _tokenize(rec.memory))
            score = overlap / max(len(q_tokens), 1)
            if score >= threshold:
                scored.append((score, rec))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [
            {"id": r.id, "memory": r.memory, "score": round(s, 3), "metadata": r.metadata}
            for s, r in scored[:top_k]
        ]

    def get_all(self, *, filters: dict[str, str] | None = None) -> list[dict[str, Any]]:
        filters = filters or {}
        if not any(filters.get(k) for k in ("user_id", "agent_id", "run_id")):
            raise ValueError("get_all requires at least one entity filter")
        results = []
        for rec in self._records.values():
            if filters.get("user_id") and rec.user_id != filters["user_id"]:
                continue
            if filters.get("agent_id") and rec.agent_id != filters["agent_id"]:
                continue
            results.append({"id": rec.id, "memory": rec.memory, "metadata": rec.metadata})
        return results

    def update(self, memory_id: str, data: str) -> dict[str, str]:
        if memory_id not in self._records:
            return {"event": "ERROR", "message": "not found"}
        self._records[memory_id].memory = data
        return {"event": "UPDATE", "id": memory_id}

    def delete(self, memory_id: str) -> dict[str, str]:
        if memory_id in self._records:
            del self._records[memory_id]
            return {"event": "DELETE", "id": memory_id}
        return {"event": "ERROR", "message": "not found"}


MEM0 = ShopCoMem0()
print("ShopCoMem0 client ready.")

---

## 4. `add()` — Inferred vs Raw Storage

In [ ]:
messages = [
    {"role": "user", "content": "I'm vegetarian and allergic to nuts."},
    {"role": "assistant", "content": "I'll remember your dietary restrictions."},
]

inferred = MEM0.add(messages, user_id="alice", agent_id=SHOPCO_AGENT_ID, metadata={"category": "dietary"})
print("infer=True:", inferred)

raw = MEM0.add(
    "Raw log: customer called about ORD-1001",
    user_id="alice", agent_id=SHOPCO_AGENT_ID, infer=False,
    metadata={"category": "audit_log"},
)
print("infer=False:", raw)

---

## 5. `search()` — v3 Filters API

In [ ]:
hits = MEM0.search(
    "dietary restrictions and allergies",
    filters={"user_id": "alice", "agent_id": SHOPCO_AGENT_ID},
    top_k=5,
    threshold=0.0,
)
print("Search hits:")
for h in hits:
    print(f"  [{h['score']}] {h['memory'][:70]}")

---

## 6. `get_all()` and Memory Updates

In [ ]:
all_alice = MEM0.get_all(filters={"user_id": "alice"})
print(f"Alice has {len(all_alice)} memories")

if all_alice:
    mid = all_alice[0]["id"]
    print("Update:", MEM0.update(mid, "User is vegetarian — strictly no meat or fish"))
    print("After update:", MEM0.search("vegetarian", filters={"user_id": "alice"})[0]["memory"])

---

## 7. Graph State — Thread Scratch + Mem0 Context

In [ ]:
class Mem0AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    order_id: str
    customer_name: str
    turn_count: int
    mem0_context: str
    mem0_trace: list[str]


def get_config_ids(config: RunnableConfig) -> tuple[str, str]:
    cfg = config.get("configurable", {})
    return cfg.get("user_id", "anonymous"), cfg.get("thread_id", "default")


def extract_entities(state: Mem0AgentState) -> dict[str, Any]:
    last = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    if not last:
        return {"turn_count": state.get("turn_count", 0)}
    text = str(last.content)
    updates: dict[str, Any] = {"turn_count": state.get("turn_count", 0) + 1}
    m = re.search(r"ORD-[0-9]{4}", text.upper())
    if m:
        updates["order_id"] = m.group(0)
    name_m = re.search(r"my name is ([A-Za-z]+)", text, re.I)
    if name_m:
        updates["customer_name"] = name_m.group(1)
    return updates

---

## 8. Load Mem0 Context — Start of Each Turn

In [ ]:
def load_mem0_node(state: Mem0AgentState, config: RunnableConfig, *, mem0: ShopCoMem0) -> dict[str, Any]:
    user_id, _ = get_config_ids(config)
    last = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    query = str(last.content) if last else "user preferences"
    hits = mem0.search(
        query,
        filters={"user_id": user_id, "agent_id": SHOPCO_AGENT_ID},
        top_k=5,
        threshold=0.0,
    )
    if not hits:
        hits = mem0.get_all(filters={"user_id": user_id})[:5]
    lines = [f"[{h.get('score', 0)}] {h['memory']}" for h in hits]
    ctx = "\n".join(lines) if lines else "(no Mem0 memories)"
    return {
        "mem0_context": ctx,
        "mem0_trace": state.get("mem0_trace", []) + [f"load:{len(hits)} hits"],
    }

---

## 9. Respond Node — Uses Mem0 + Thread State

In [ ]:
def respond_node(state: Mem0AgentState, config: RunnableConfig) -> dict[str, Any]:
    last = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    q = str(last.content).lower() if last else ""
    name = state.get("customer_name") or "there"
    order_id = state.get("order_id", "")
    mem = state.get("mem0_context", "")

    if "return" in q:
        body = POLICY_SNIPPETS["returns"]
    elif "shipping" in q:
        body = POLICY_SNIPPETS["shipping"]
    elif "status" in q or ("it" in q and order_id):
        row = ORDERS_DB.get(order_id, {})
        body = f"Order {order_id or '?'} status: {row.get('status', 'unknown')}."
    elif "what do you know" in q or "my preferences" in q or "remember about me" in q:
        body = f"From Mem0 long-term memory:\n{mem}"
    elif "remember" in q or "prefer" in q:
        body = "Got it — I'll store that in Mem0 for future sessions."
    else:
        body = "How can I help with your ShopCo order or policy?"

    if "concise" in mem.lower() and "what do you know" not in q:
        body = body.split(".")[0] + "."

    if mem and mem != "(no Mem0 memories)" and "dietary" in q:
        body += f"\n\nRelevant memories:\n{mem}"

    return {"messages": [AIMessage(content=f"Hi {name}, {body}")]}

---

## 10. Persist to Mem0 — End of Each Turn

In [ ]:
def persist_mem0_node(state: Mem0AgentState, config: RunnableConfig, *, mem0: ShopCoMem0) -> dict[str, Any]:
    """Mirror Mem0 add() after each turn — extraction runs on conversation slice."""
    user_id, thread_id = get_config_ids(config)
    messages: list[dict[str, str]] = []
    for m in state["messages"][-4:]:
        if isinstance(m, HumanMessage):
            messages.append({"role": "user", "content": str(m.content)})
        elif isinstance(m, AIMessage):
            messages.append({"role": "assistant", "content": str(m.content)})
    if not messages:
        return {}

    result = mem0.add(
        messages,
        user_id=user_id,
        agent_id=SHOPCO_AGENT_ID,
        run_id=thread_id,
        metadata={"source": "shopco_support", "turn": state.get("turn_count", 0)},
        infer=True,
    )
    n = len(result.get("results", []))
    if n:
        return {"mem0_trace": state.get("mem0_trace", []) + [f"persist:ADD x{n}"]}
    return {}

---

## 11. Compile LangGraph — Checkpointer + Mem0 Client

In [ ]:
def build_mem0_graph(mem0: ShopCoMem0, checkpointer: MemorySaver | None = None):
    def load_node(state: Mem0AgentState, config: RunnableConfig) -> dict[str, Any]:
        return load_mem0_node(state, config, mem0=mem0)

    def persist_node(state: Mem0AgentState, config: RunnableConfig) -> dict[str, Any]:
        return persist_mem0_node(state, config, mem0=mem0)

    builder = StateGraph(Mem0AgentState)
    builder.add_node("load_mem0", load_node)
    builder.add_node("extract", extract_entities)
    builder.add_node("respond", respond_node)
    builder.add_node("persist_mem0", persist_node)

    builder.add_edge(START, "load_mem0")
    builder.add_edge("load_mem0", "extract")
    builder.add_edge("extract", "respond")
    builder.add_edge("respond", "persist_mem0")
    builder.add_edge("persist_mem0", END)

    return builder.compile(checkpointer=checkpointer)


CHECKPOINTER = MemorySaver()
GRAPH = build_mem0_graph(MEM0, checkpointer=CHECKPOINTER)
print("Mem0 graph compiled with MemorySaver")

---

## 12. Invoke Helper

In [ ]:
def send_message(
    graph,
    user_id: str,
    thread_id: str,
    text: str,
    state: Mem0AgentState | None = None,
) -> Mem0AgentState:
    config: RunnableConfig = {"configurable": {"user_id": user_id, "thread_id": thread_id}}
    payload: dict[str, Any] = {"messages": [HumanMessage(content=text)]}
    if state:
        for key in ("order_id", "customer_name", "turn_count", "mem0_trace"):
            if state.get(key):
                payload[key] = state[key]
    return graph.invoke(payload, config=config)


def last_ai(state: Mem0AgentState) -> str:
    for m in reversed(state["messages"]):
        if isinstance(m, AIMessage):
            return str(m.content)
    return ""

---

## 13. Demo — Store Preference in Thread 1

In [ ]:
FRESH_MEM0 = ShopCoMem0()
FRESH_GRAPH = build_mem0_graph(FRESH_MEM0, checkpointer=MemorySaver())

s1 = send_message(FRESH_GRAPH, "alice", "chat-mon", "My name is Alice. I prefer email contact for support.")
print(last_ai(s1))
print("Trace:", s1.get("mem0_trace", []))
print("Mem0 store:", [m["memory"][:50] for m in FRESH_MEM0.get_all(filters={"user_id": "alice"})])

---

## 14. Cross-Thread Recall — New Chat, Same User

In [ ]:
s2 = send_message(FRESH_GRAPH, "alice", "chat-fri", "What do you remember about me?")
print(last_ai(s2))
print("\nMem0 context loaded:")
print(s2.get("mem0_context", ""))

---

## 15. User Isolation and `agent_id` Scoping

In [ ]:
s_bob = send_message(FRESH_GRAPH, "bob", "bob-1", "What are my contact preferences?")
print("Bob:", last_ai(s_bob))
print("Bob memories:", FRESH_MEM0.get_all(filters={"user_id": "bob"}))

# Different agent_id — sales bot should not see support memories
FRESH_MEM0.add("User wants enterprise pricing", user_id="alice", agent_id="shopco-sales-v1")
support_hits = FRESH_MEM0.search("pricing", filters={"user_id": "alice", "agent_id": SHOPCO_AGENT_ID})
sales_hits = FRESH_MEM0.search("pricing", filters={"user_id": "alice", "agent_id": "shopco-sales-v1"})
print(f"\nSupport agent hits: {len(support_hits)}, Sales agent hits: {len(sales_hits)}")

---

## 16. Metadata Filtering Pattern

In [ ]:
FRESH_MEM0.add(
    "Customer escalated billing dispute for ORD-1001",
    user_id="alice", agent_id=SHOPCO_AGENT_ID,
    metadata={"category": "escalation", "order_id": "ORD-1001"},
    infer=False,
)

all_alice = FRESH_MEM0.get_all(filters={"user_id": "alice"})
escalations = [m for m in all_alice if m.get("metadata", {}).get("category") == "escalation"]
print(f"Escalation memories: {len(escalations)}")
print(escalations[0]["memory"] if escalations else "none")

---

## 17. Integration Patterns

| Pattern | How |
|---------|-----|
| **Load-then-respond** | `search()` at turn start → inject into system prompt |
| **Persist-after-respond** | `add()` on conversation slice after each turn |
| **Tool-based** | Expose Mem0 as agent tools (like LangMem hot path) |
| **Background batch** | `add()` full transcript when session ends |
| **Managed platform** | Swap `ShopCoMem0` for `MemoryClient(api_key=...)` |

---

## 18. Eval Harness

In [ ]:
EVAL = [
    ("alice has memories", lambda: len(FRESH_MEM0.get_all(filters={"user_id": "alice"})) >= 1),
    ("bob isolated", lambda: len(FRESH_MEM0.get_all(filters={"user_id": "bob"})) == 0),
    ("cross-thread recall", lambda: "email" in s2.get("mem0_context", "").lower()),
    ("agent_id scoping", lambda: len(sales_hits) >= 1 and len(support_hits) == 0),
    ("metadata stored", lambda: len(escalations) >= 1),
]

passed = sum(int(fn()) for _, fn in EVAL)
for label, fn in EVAL:
    print(f"{'✓' if fn() else '✗'} {label}")
print(f"\nEval: {passed}/{len(EVAL)}")

---

## 19. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| `user_id` on `search()` top-level (v3) | `ValueError` | Use `filters={"user_id": ...}` |
| No `user_id` on `add()` | Memories not retrievable | Always scope `add()` |
| Same `agent_id` for all bots | Sales sees support secrets | Separate `agent_id` per agent |
| `infer=False` for preferences | Raw chat logs, poor search | Use `infer=True` for facts |
| Only Mem0, no checkpointer | Thread state lost | Keep `MemorySaver` for active chat |
| No `threshold=0.0` in v3 | Low-score results filtered | Tune threshold explicitly |

---

## 20. Production Checklist

- [ ] `pip install mem0ai` (OSS) or `MemoryClient` with API key (platform)
- [ ] Consistent `user_id` from auth system — not `thread_id`
- [ ] `agent_id` per agent role (support, sales, internal)
- [ ] v3 `filters` on `search()` / `get_all()`
- [ ] `infer=True` for facts; `infer=False` for audit logs only
- [ ] `metadata` for category, order_id, source system
- [ ] LangGraph checkpointer for thread + Mem0 for long-term
- [ ] Memory governance: conflicts, decay, audit trails

---

## 21. Optional Live Mem0 OSS

Set `USE_LIVE_LLM = True` to use the real Mem0 `Memory` class with OpenAI extraction.

In [ ]:
def demo_live_mem0():
    if not USE_LIVE_LLM:
        print("Set USE_LIVE_LLM = True to run live Mem0.")
        return

    from mem0 import Memory

    config = {
        "vector_store": {"provider": "qdrant", "config": {"on_disk": False}},
        "llm": {"provider": "openai", "config": {"model": "gpt-4o-mini", "api_key": OPENAI_API_KEY}},
        "embedder": {"provider": "openai", "config": {"model": "text-embedding-3-small", "api_key": OPENAI_API_KEY}},
    }
    m = Memory.from_config(config)
    messages = [{"role": "user", "content": "Remember I prefer email for ShopCo support."}]
    result = m.add(messages, user_id="alice-live", agent_id=SHOPCO_AGENT_ID)
    print("add:", result)
    hits = m.search("contact preference", filters={"user_id": "alice-live"}, threshold=0.0)
    print("search:", hits)


demo_live_mem0()

---

## 22. Quiz

1. Where do entity IDs go in Mem0 v3 — `add()` vs `search()`?
2. What does `infer=True` do on `add()`?
3. Why use separate `agent_id` values for support vs sales?
4. How does Mem0 differ from LangGraph's `BaseStore` + LangMem?
5. Why keep a checkpointer when Mem0 already stores long-term facts?

---

## 23. Summary

**Mem0** provides a dedicated long-term memory layer with built-in extraction and hybrid search:

1. **`add()`** — store inferred facts or raw logs, scoped by `user_id` / `agent_id`
2. **`search()`** — retrieve with v3 `filters` and tunable `threshold` / `top_k`
3. **LangGraph integration** — load at turn start, persist after respond
4. **Isolation** — per-user and per-agent memory boundaries

ShopCo Support demonstrates cross-thread recall with Mem0 while thread scratch lives in the checkpointer. Choose **LangMem** for tight LangGraph coupling or **Mem0** for a managed, framework-agnostic memory service.